# Projet 4 — Partie 2 : Conteneurisation avec Docker

**Objectif :** rendre la migration CSV → MongoDB reproductible avec Docker, sécuriser la base et vérifier le fonctionnement du déploiement.

Cette partie reprend les manipulations effectuées après le premier notebook `P4_analyse_dataset_medical.ipynb`, qui s'arrête aux opérations CRUD.

> **Confidentialité :** aucun mot de passe réel, aucune donnée médicale individuelle et aucun identifiant personnel ne doivent apparaître dans ce notebook ou dans GitHub. Le CSV et le fichier `.env` restent en local.

**Mode d'emploi :** les blocs PowerShell, Dockerfile, YAML et JavaScript sont des extraits à placer dans leurs fichiers ou à exécuter dans un terminal. Ils ne sont **pas** des cellules Python exécutables dans Jupyter. Seule la cellule de test PyMongo est exécutable dans le notebook.

## 1. Point de départ et objectifs

La partie 1 a permis de :

- Analyser le CSV : **55 500 lignes et 15 colonnes**.
- Identifier puis supprimer, selon la règle de nettoyage retenue, **534 doublons exacts**.
- Obtenir **54 966 hospitalisations** à migrer.
- Définir quatre sous-documents MongoDB : `patient`, `medical`, `hospitalisation` et `facturation`.
- Tester la migration initiale et les opérations CRUD sur MongoDB en local.

La partie 2 a pour objectifs de :

1. Conteneuriser MongoDB et le script Python `migration.py`.
2. Conserver les données avec un volume persistant.
3. Vérifier la disponibilité de MongoDB avant de lancer la migration.
4. Séparer trois niveaux d'autorisation : administrateur, analyste et Data Engineer.
5. Automatiser la création des utilisateurs lors d'un déploiement neuf.
6. Documenter les tests et versionner le travail sur GitHub.

> **Portée :** Docker Compose rend notre déploiement portable et reproductible. Il ne rend pas, à lui seul, MongoDB distribué ou horizontalement scalable.

## 2. Quelques notions Docker

- **Image** : modèle utilisé pour construire un conteneur (par exemple `mongo:8.0` ou une image Python 3.12).
- **Conteneur** : environnement isolé qui exécute un service ou un script.
- **Dockerfile** : fichier décrivant comment construire notre image Python.
- **Docker Compose** : fichier YAML qui définit les services et leurs connexions.
- **Volume** : stockage séparé du cycle de vie du conteneur, utilisé pour conserver les données MongoDB.

Notre architecture :

```text
CSV local (data/healthcare_dataset.csv)
            |
            | volume en lecture seule
            v
   Conteneur Python (migration)
            |
            | réseau Docker : mongodb:27017
            v
    Conteneur MongoDB (mongodb)
            |
            v
     Volume nommé mongo_data
```

Depuis **Windows**, nous accédons à ce MongoDB par `127.0.0.1:27018`. Entre les **conteneurs**, Python utilise `mongodb:27017`. Cela évite un conflit avec MongoDB installé directement sur Windows, qui utilise déjà le port `27017`.

## 3. Organisation du projet

```text
Project_4/
├── data/
│   └── healthcare_dataset.csv          # non publié sur GitHub
├── docker/
│   └── init_users.js                   # création des comptes au premier lancement
├── reports/
│   └── qualite_avant_migration.json
├── P4_analyse_dataset_medical.ipynb   # partie 1
├── P4_conteneurisation_docker.ipynb   # partie 2
├── P4_rapport_analyse_et_schema.md
├── migration.py
├── Dockerfile
├── docker-compose.yml
├── docker-compose.test.yml            # instance MongoDB temporaire de test
├── requirements.txt
├── .dockerignore
├── .gitignore
├── .env                               # secrets locaux, ignorés par Git
└── README.md
```

Les données CSV sont **montées** dans le conteneur, pas intégrées à l'image Python. La base Docker est distincte de la base MongoDB installée sur Windows.

Le fichier `.env.example` sera ajouté lors de la finalisation de la documentation.

## 4. Dépendances Python — `requirements.txt`

Les bibliothèques utilisées par le script sont :

| Bibliothèque | Utilité |
|---|---|
| `pandas` | Charger, nettoyer et transformer le CSV |
| `pymongo` | Communiquer avec MongoDB |
| `python-dotenv` | Lire les variables du fichier `.env` lors d'une exécution locale |

Nous avons créé `requirements.txt` avec les **versions réellement installées** sur notre machine, au format `pandas==X.Y.Z`.

Commande utilisée dans PowerShell pour connaître les versions :

```powershell
python -m pip freeze | Select-String "^(pandas|pymongo|python-dotenv)=="
```

`os` et `pathlib` font partie de Python ; ils n'ont pas besoin de figurer dans `requirements.txt`.

## 5. Construire l'image Python — `Dockerfile`

Notre Dockerfile est simple :

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY migration.py .

CMD ["python", "migration.py"]
```

**Explication :**

- `FROM` : choisit une image Python légère.
- `WORKDIR` : définit le répertoire de travail du conteneur.
- `COPY requirements.txt` + `RUN` : installent les dépendances.
- `COPY migration.py` : ajoute le script à l'image.
- `CMD` : précise la commande exécutée au lancement.

Le CSV n'est **pas inclus dans l'image** : Docker Compose le rendra accessible par un volume monté en lecture seule.

## 6. Protéger les données et les mots de passe

Dans `.gitignore` :

```gitignore
data/
.env
.env.*
!.env.example
.ipynb_checkpoints/
__pycache__/
.venv/
```

Dans `.dockerignore` :

```gitignore
.git/
data/
.env
.env.*
.ipynb_checkpoints/
*.ipynb
reports/
__pycache__/
```

Le fichier local `.env` contient les variables nécessaires, **avec nos vrais mots de passe uniquement sur le PC** :

```dotenv
MONGO_ROOT_USER=admin
MONGO_ROOT_PASSWORD=VALEUR_A_RENSEIGNER
MONGO_MIGRATION_USER=data_engineer
MONGO_MIGRATION_PASSWORD=VALEUR_A_RENSEIGNER
MONGO_ANALYST_PASSWORD=VALEUR_A_RENSEIGNER
```

Les mots de passe des trois comptes sont distincts. Dans les extraits de ce notebook, les valeurs ci-dessus sont **des espaces réservés**, pas de véritables identifiants.

> Les variables d'environnement facilitent la configuration ; pour une architecture de production, on privilégierait une gestion dédiée des secrets plutôt qu'un simple fichier `.env`.

## 7. Configuration finale — `docker-compose.yml`

Nous déclarons deux services : le serveur MongoDB et le conteneur Python chargé de la migration.

```yaml
services:
  mongodb:
    image: mongo:8.0
    container_name: datasolutech_mongodb

    ports:
      - "127.0.0.1:27018:27017"

    environment:
      MONGO_INITDB_ROOT_USERNAME: ${MONGO_ROOT_USER}
      MONGO_INITDB_ROOT_PASSWORD: ${MONGO_ROOT_PASSWORD}
      MONGO_ANALYST_PASSWORD: ${MONGO_ANALYST_PASSWORD}
      MONGO_MIGRATION_PASSWORD: ${MONGO_MIGRATION_PASSWORD}

    volumes:
      - mongo_data:/data/db
      - ./docker/init_users.js:/docker-entrypoint-initdb.d/init_users.js:ro

    healthcheck:
      test: ["CMD", "mongosh", "--quiet", "--eval", "db.adminCommand({ ping: 1 })"]
      interval: 10s
      timeout: 5s
      retries: 5
      start_period: 20s

  migration:
    build: .

    depends_on:
      mongodb:
        condition: service_healthy

    environment:
      MONGO_HOST: mongodb
      MONGO_PORT: 27017
      MONGO_MIGRATION_USER: ${MONGO_MIGRATION_USER}
      MONGO_MIGRATION_PASSWORD: ${MONGO_MIGRATION_PASSWORD}

    volumes:
      - ./data:/app/data:ro

volumes:
  mongo_data:
```

**Explications :**

- `127.0.0.1:27018:27017` : MongoDB Docker est accessible depuis Windows sur `27018`, sans exposer le port à toutes les interfaces du PC.
- `mongo_data:/data/db` : les documents et les utilisateurs survivent à la recréation du conteneur.
- `./data:/app/data:ro` : le script lit le CSV sans pouvoir le modifier.
- `init_users.js:ro` : notre script de création des utilisateurs est accessible à MongoDB en lecture seule.
- `healthcheck` + `depends_on: condition: service_healthy` : le conteneur Python attend que MongoDB réponde.
- Les identifiants **`data_engineer`** sont transmis au conteneur de migration ; le script ne reçoit plus les identifiants administrateur.

> Le `healthcheck` teste la disponibilité du serveur, pas la capacité d'un utilisateur particulier à s'authentifier. Les comptes ne sont créés automatiquement **que lors de l'initialisation d'un nouveau volume MongoDB**.

## 8. Démarrage et contrôles Docker

Dans PowerShell, depuis `D:\jupyter_notebook\Project_4` :

```powershell
docker compose config --quiet
docker compose up -d mongodb
docker compose ps
```

- `config --quiet` vérifie la configuration YAML.
- `up -d mongodb` démarre MongoDB en arrière-plan.
- `ps` affiche son état.

Au premier démarrage, Docker a créé le réseau et le volume `project_4_mongo_data`. L'instance Windows reste sur `27017` ; MongoDB Docker est disponible sur `27018` côté Windows.

Nous avons ajouté puis appliqué le contrôle de santé avec :

```powershell
docker compose up -d --force-recreate mongodb
docker compose ps
```

Le conteneur a d'abord affiché `health: starting`. Lorsque le contrôle réussit, l'état devient `healthy`.

Pour diagnostiquer le contrôle de santé :

```powershell
docker compose logs mongodb
docker inspect datasolutech_mongodb --format '{{json .Config.Healthcheck}}'
```

Nous avions initialement obtenu `null`, ce qui signifiait que le `healthcheck` n'était pas intégré. Après correction du fichier Compose et recréation, Docker a reconnu le contrôle.

**À éviter sur la base principale :** `docker compose down -v` supprimerait son volume et donc les données conservées dedans.

## 9. Vérifier MongoDB à l'intérieur du conteneur

Dans **PowerShell** :

```powershell
docker exec -it datasolutech_mongodb mongosh -u admin -p --authenticationDatabase admin
```

Le paramètre `-p` demande le mot de passe sans l'inclure dans la commande.

Une fois connecté à `mongosh` :

```javascript
db.adminCommand({ ping: 1 })
```

**Résultat attendu :** `{ ok: 1 }`.

Ce test a permis de confirmer que MongoDB répond et que le compte administrateur est utilisable. Nous avons ensuite quitté le shell avec `exit`.

## 10. Tester la connexion Python → MongoDB Docker

Sur **Windows**, le serveur MongoDB du conteneur répond à l'adresse `localhost:27018`.

La cellule suivante teste uniquement le `ping` ; elle ne lit ni ne modifie les données médicales. Le fichier `.env` doit se trouver dans le dossier depuis lequel Jupyter exécute le notebook.

In [ ]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv(".env")

client_docker = MongoClient(
    "mongodb://localhost:27018/",
    username=os.getenv("MONGO_ROOT_USER"),
    password=os.getenv("MONGO_ROOT_PASSWORD"),
    authSource="admin"
)

try:
    client_docker.admin.command("ping")
    print("Connexion Python → MongoDB Docker : OK")
finally:
    client_docker.close()

## 11. Configurer `migration.py` pour Windows et Docker

Nous utilisons les mêmes variables pour l'adresse du serveur, mais **le compte `data_engineer`** pour l'authentification.

Extrait de `migration.py` (les imports `os`, `load_dotenv` et `MongoClient` sont déjà nécessaires dans le script) :

```python
load_dotenv(".env")

host = os.getenv("MONGO_HOST", "localhost")
port = int(os.getenv("MONGO_PORT", "27018"))

client = MongoClient(
    host=host,
    port=port,
    username=os.getenv("MONGO_MIGRATION_USER"),
    password=os.getenv("MONGO_MIGRATION_PASSWORD"),
    authSource="healthcare"
)

client.admin.command("ping")
```

| Lieu d'exécution | Hôte et port | Pourquoi ? |
|---|---|---|
| Windows | `localhost:27018` | Port exposé par le conteneur MongoDB |
| Conteneur `migration` | `mongodb:27017` | Nom du service Compose et port interne |

`authSource="healthcare"` est nécessaire car le compte `data_engineer` a été créé **dans la base `healthcare`**. Le compte administrateur est quant à lui créé dans `admin`.

Les autres étapes de `migration.py` ne changent pas : lecture du CSV, suppression des doublons, conversion des dates, construction des sous-documents, insertion et vérifications. La connexion est fermée avec `client.close()` **après** les dernières opérations MongoDB.

## 12. Exécuter et vérifier la migration dans Docker

Dans PowerShell :

```powershell
docker compose run --rm --build migration
```

- `--build` : construit l'image Python depuis le Dockerfile.
- `run` : lance un conteneur de migration temporaire.
- `--rm` : le supprime lorsque le script est terminé.

**Résultats des essais réalisés :**

- **54 966 documents** ont été importés dans `healthcare.hospitalisations` sur MongoDB Docker.
- Le script a vérifié le **nombre** de documents et la présence des **quatre sous-documents**.
- Une exécution ultérieure n'a pas relancé l'insertion, car la collection contenait déjà le nombre attendu.
- La connexion du script a ensuite été testée avec le compte **`data_engineer`** plutôt qu'avec `admin`.

**Limite du contrôle :** l'égalité entre deux nombres de documents ne suffit pas à garantir que toutes les valeurs sont identiques. Des contrôles supplémentaires, notamment la comparaison des contenus source/cible, restent utiles pour une validation exhaustive.

Le conteneur Python n'a pas vocation à fonctionner en permanence : c'est un travail de migration ponctuel.

## 13. Authentification et trois rôles MongoDB

Nous avons défini trois comptes, conformément au principe du **moindre privilège** :

| Profil | Utilisateur | Rôle MongoDB | Base du compte | Droits |
|---|---|---|---|---|
| Administrateur | `admin` | `root` | `admin` | Administration globale |
| Analyste | `analyste` | `read` | `healthcare` | Consultation uniquement |
| Data Engineer | `data_engineer` | `readWrite` | `healthcare` | Lecture et écriture des documents |

L'administrateur est créé grâce aux variables `MONGO_INITDB_ROOT_*`. Les deux comptes métier ont **d'abord été créés manuellement**, puis leur création a été **automatisée** via `init_users.js` pour les nouvelles installations.

> Changer les variables `MONGO_INITDB_ROOT_*` ne modifie pas le mot de passe d'un administrateur déjà enregistré dans un volume MongoDB existant.

## 14. Première création manuelle des deux comptes

Après connexion avec le compte `admin` à `mongosh`, nous avons travaillé dans la base `healthcare` :

```javascript
use healthcare

db.createUser({
    user: "analyste",
    pwd: passwordPrompt(),
    roles: [{ role: "read", db: "healthcare" }]
})

db.createUser({
    user: "data_engineer",
    pwd: passwordPrompt(),
    roles: [{ role: "readWrite", db: "healthcare" }]
})

db.getUsers()
```

`passwordPrompt()` permet de saisir un mot de passe sans l'inscrire en clair dans le code.

Cette création manuelle nous a permis de découvrir les commandes MongoDB et de tester les droits. **La création sur une instance neuve est désormais automatisée** (voir section 18). Les comptes déjà présents dans notre volume principal ne sont pas recréés.

## 15. Vérifier les permissions sans toucher aux hospitalisations

Nous avons utilisé une collection distincte, `tests_permissions`, pour les essais.

**Connexion analyste depuis PowerShell :**

```powershell
docker exec -it datasolutech_mongodb mongosh -u analyste -p --authenticationDatabase healthcare
```

Dans `mongosh` :

```javascript
use healthcare
db.hospitalisations.countDocuments({})
db.tests_permissions.insertOne({ test: "analyste" })
```

Le comptage doit fonctionner ; l'insertion doit échouer avec une erreur d'autorisation.

**Connexion Data Engineer :**

```powershell
docker exec -it datasolutech_mongodb mongosh -u data_engineer -p --authenticationDatabase healthcare
```

Les opérations `insertOne`, `findOne`, `updateOne` et `deleteOne` ont été testées sur un document temporaire de `tests_permissions`, puis ce document a été supprimé.

Cela vérifie que l'analyste est en **lecture seule** et que le Data Engineer dispose des droits **CRUD** sur `healthcare`.

## 16. Modifier les mots de passe sans changer les droits

Depuis une session `mongosh` administrateur, dans `healthcare` :

```javascript
use healthcare
db.changeUserPassword("analyste", passwordPrompt())
db.changeUserPassword("data_engineer", passwordPrompt())
```

Ces commandes permettent de changer les mots de passe sans recréer les utilisateurs, sans modifier leurs rôles et sans toucher aux documents médicaux.

Les mots de passe ne doivent apparaître **ni dans GitHub ni dans le notebook**.

## 17. Utiliser un compte dédié pour la migration

Nous avons remplacé les identifiants administrateur par ceux du compte `data_engineer` dans `migration.py` et dans le service `migration` de Docker Compose.

Dans le `.env` local :

```dotenv
MONGO_MIGRATION_USER=data_engineer
MONGO_MIGRATION_PASSWORD=VALEUR_A_RENSEIGNER
```

Dans `migration.py` :

```python
client = MongoClient(
    host=host,
    port=port,
    username=os.getenv("MONGO_MIGRATION_USER"),
    password=os.getenv("MONGO_MIGRATION_PASSWORD"),
    authSource="healthcare"
)
```

Dans le service `migration` de `docker-compose.yml`, les variables `MONGO_MIGRATION_USER` et `MONGO_MIGRATION_PASSWORD` sont transmises au conteneur Python.

**Validation réalisée :** `docker compose run --rm --build migration` a fonctionné avec ce compte. Comme les données étaient déjà importées, ce test confirmait la connexion et la lecture ; les droits d'écriture avaient été vérifiés auparavant avec les tests CRUD.

**Justification :** le script peut travailler sur `healthcare` sans obtenir les permissions `root` de l'administrateur.

## 18. Automatiser la création des utilisateurs — `docker/init_users.js`

La création manuelle convient pour découvrir MongoDB, mais **un déploiement neuf** doit aussi installer les comptes automatiquement.

Nous avons créé le fichier `docker/init_users.js` :

```javascript
const database = db.getSiblingDB("healthcare");

const analystPassword = process.env.MONGO_ANALYST_PASSWORD;
const engineerPassword = process.env.MONGO_MIGRATION_PASSWORD;

if (!analystPassword || !engineerPassword) {
    throw new Error("Mots de passe MongoDB manquants");
}

database.createUser({
    user: "analyste",
    pwd: analystPassword,
    roles: [{ role: "read", db: "healthcare" }]
});

database.createUser({
    user: "data_engineer",
    pwd: engineerPassword,
    roles: [{ role: "readWrite", db: "healthcare" }]
});

print("Utilisateurs MongoDB créés avec succès");
```

**Explications :**

- `db.getSiblingDB("healthcare")` sélectionne la base où créer les comptes.
- `process.env` lit les mots de passe transmis par Docker, sans les écrire en clair dans le fichier.
- `createUser()` associe chaque compte à un rôle (`read` ou `readWrite`).
- Le message final permet de confirmer l'exécution dans les logs.

L'utilisateur administrateur est créé séparément par l'image officielle MongoDB avec `MONGO_INITDB_ROOT_USERNAME` et `MONGO_INITDB_ROOT_PASSWORD`.

## 19. Intégrer l'initialisation automatique à Docker Compose

Dans le service `mongodb` de `docker-compose.yml`, nous avons ajouté :

```yaml
environment:
  MONGO_INITDB_ROOT_USERNAME: ${MONGO_ROOT_USER}
  MONGO_INITDB_ROOT_PASSWORD: ${MONGO_ROOT_PASSWORD}
  MONGO_ANALYST_PASSWORD: ${MONGO_ANALYST_PASSWORD}
  MONGO_MIGRATION_PASSWORD: ${MONGO_MIGRATION_PASSWORD}

volumes:
  - mongo_data:/data/db
  - ./docker/init_users.js:/docker-entrypoint-initdb.d/init_users.js:ro
```

Cet extrait illustre **les parties `environment` et `volumes` du service `mongodb`** ; pour le fichier Compose complet, voir la section 7.

Les scripts placés dans `/docker-entrypoint-initdb.d/` sont exécutés lors de la **première initialisation** d'un volume MongoDB vide.

Notre volume principal `project_4_mongo_data` contient déjà les 54 966 documents et les trois comptes : le script d'initialisation n'est donc **pas rejoué** simplement en redémarrant le conteneur.

**Conséquence :** nous avons testé cette automatisation sur une instance temporaire, sans supprimer le volume principal.

## 20. Tester l'initialisation sur une base MongoDB indépendante

Nous avons créé un fichier `docker-compose.test.yml` pour ne pas modifier l'instance principale :

```yaml
services:
  mongodb_test:
    image: mongo:8.0

    environment:
      MONGO_INITDB_ROOT_USERNAME: ${MONGO_ROOT_USER}
      MONGO_INITDB_ROOT_PASSWORD: ${MONGO_ROOT_PASSWORD}
      MONGO_ANALYST_PASSWORD: ${MONGO_ANALYST_PASSWORD}
      MONGO_MIGRATION_PASSWORD: ${MONGO_MIGRATION_PASSWORD}

    volumes:
      - mongo_test_data:/data/db
      - ./docker/init_users.js:/docker-entrypoint-initdb.d/init_users.js:ro

volumes:
  mongo_test_data:
```

**Démarrage et consultation des logs, dans PowerShell :**

```powershell
docker compose -p datasolutech_test -f docker-compose.test.yml up -d
docker compose -p datasolutech_test -f docker-compose.test.yml logs mongodb_test
```

Nous avons retrouvé le message :

```text
Utilisateurs MongoDB créés avec succès
```

Cela indique que notre script a été exécuté. Nous avons ensuite vérifié les comptes et rôles dans `mongosh` :

```powershell
docker compose -p datasolutech_test -f docker-compose.test.yml exec mongodb_test mongosh -u admin -p --authenticationDatabase admin
```

```javascript
use healthcare
db.getUsers()  // analyste : read ; data_engineer : readWrite
use admin
db.getUser("admin")  // rôle root
```

**Résultat constaté :** les trois comptes étaient présents avec les rôles attendus. Le test démontre la reproductibilité de la **création des utilisateurs** lors d'une initialisation neuve, sans supposer que l'ensemble du déploiement a été testé sur une autre machine.

## 21. Supprimer l'instance de test sans toucher à la base principale

Après vérification, nous avons fermé la session `mongosh` (`exit`) puis retiré l'environnement temporaire avec :

```powershell
docker compose -p datasolutech_test -f docker-compose.test.yml down -v
```

L'option `-p datasolutech_test` désigne **uniquement le projet de test**. `down -v` supprime son conteneur, son réseau et son **volume de test**.

**Attention :** ne pas exécuter `docker compose down -v` sur le projet principal, car cela supprimerait son volume et les données MongoDB.

Contrôle possible après le test :

```powershell
docker compose ps
```

Notre service `datasolutech_mongodb` et son volume `project_4_mongo_data` restent indépendants de l'environnement de test.

## 22. Bilan des tests et suite du projet

| Vérification | Résultat de nos essais |
|---|---|
| MongoDB 8.0 dans Docker | Démarrage réussi |
| Volume MongoDB persistant | Créé et conservé |
| Accès depuis Windows sur `127.0.0.1:27018` | Testé |
| Authentification administrateur | Testée |
| Conteneur Python exécutant `migration.py` | Testé |
| Importation dans MongoDB Docker | **54 966 documents** |
| Contrôles de nombre et de structure | Réussis |
| Healthcheck | Configuré et reconnu après correction |
| Analyste `read` et Data Engineer `readWrite` | Permissions CRUD testées |
| Migration avec `data_engineer` | Testée |
| Création automatique des comptes sur volume neuf | Testée dans l'instance temporaire |
| Vérification des trois rôles sur l'instance temporaire | Réussie |
| Nettoyage du conteneur/volume de test | Effectué sans toucher à la base principale |
| Fichiers Docker, scripts et notebook | Versionnés sur GitHub, selon les commits réalisés |

### Choix techniques justifiés

- **Sous-documents MongoDB :** les informations d'une hospitalisation restent regroupées.
- **Docker Compose :** la configuration des services est réutilisable sur un autre environnement.
- **Volume nommé :** les données survivent au remplacement du conteneur MongoDB.
- **Healthcheck :** évite de lancer la migration lorsque MongoDB ne répond pas encore.
- **Trois rôles :** sépare les usages et limite les droits selon le besoin.
- **Initialisation automatique :** évite les créations manuelles d'utilisateurs lors d'un nouveau déploiement.

### Points qu'il reste à finaliser

2. Compléter le **`README.md`** : installation, lancement, sécurité, schéma des documents et procédure de vérification.
3. Compléter les tests automatisés de contenu et les vérifications de réinstallation.
4. Préparer la comparaison des services AWS, puis la présentation finale.

**Limite importante :** Docker Compose seul n'assure pas la scalabilité horizontale de MongoDB. Elle demanderait une architecture distribuée et des choix de déploiement supplémentaires.

**Avant chaque `git push` :** vérifier `git status`, conserver `data/` et `.env` hors du dépôt, et ne pas publier de résultats Jupyter contenant des données médicales individuelles.

### Gestion des variables d'environnement

Pour sécuriser notre déploiement, les identifiants MongoDB sont stockés dans un fichier `.env`, exclu du dépôt GitHub.

Nous créons également un fichier `.env.example` contenant les noms des variables nécessaires, mais aucun véritable mot de passe.

Ce fichier sert de modèle pour reproduire notre configuration sur une autre machine.

L'utilisateur devra copier `.env.example` vers `.env`, puis renseigner ses propres identifiants avant de démarrer les conteneurs.

Cette méthode permet de partager la configuration du projet sans divulguer les informations d'authentification.

### Automatisation des index MongoDB

Afin de rendre notre déploiement reproductible, nous avons intégré la création des index directement dans le script `migration.py`.

Trois index sont créés sur les champs `medical.pathologie`, `hospitalisation.date_admission` et `hospitalisation.hopital`.

Ces index permettent d'optimiser certaines recherches dans notre collection `hospitalisations`.

La méthode `create_index()` peut être exécutée plusieurs fois sans recréer les index déjà existants avec la même configuration.

Nous vérifions leur présence avec `list_indexes()`.

Cette automatisation garantit que les index prévus sont créés lors de la migration sur une nouvelle instance MongoDB.